# NHL Interactive Shot Map

**Data:** 7 seasons (2017-18 → 2023-24) | 568k shots

This notebook provides:
1. **2D Interactive Shot Map** — rink overlay, filter by season/team/situation, color by xG/type/outcome, toggle scatter ↔ hexbin density ↔ goal-rate heatmap
2. **3D xG Danger Surface** — xG probability rendered as a terrain across the offensive zone ice surface
3. **3D Shot Scatter** — individual shots plotted in 3D with z = xG value, colored by outcome

In [ ]:
!pip install plotly ipywidgets scipy pyarrow -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy.ndimage import gaussian_filter
from scipy.stats import binned_statistic_2d
import ipywidgets as widgets
from IPython.display import display

DATA_DIR  = Path('../data/raw/nhl_shots')
PARQUET   = DATA_DIR / 'nhl_shots_combined.parquet'
PRED_PATH = DATA_DIR / 'xg_predictions.parquet'

# ── Check data exists ────────────────────────────────────────────────────────
if not PARQUET.exists():
    import subprocess, sys
    print('Data not found — running download script...')
    r = subprocess.run(['python', '../scripts/download_nhl_shots.py'], capture_output=False)
    if r.returncode != 0:
        raise RuntimeError('Download failed.')
else:
    print(f'Data found: {PARQUET}')

# ── Load & normalize ─────────────────────────────────────────────────────────
df_raw  = pd.read_parquet(PARQUET)
df_pred = pd.read_parquet(PRED_PATH) if PRED_PATH.exists() else None

# Normalize shot type labels across two source datasets
SHOT_TYPE_MAP = {
    'Wrist Shot': 'Wrist',   'wrist': 'Wrist',   'Wrist': 'Wrist',
    'Snap Shot':  'Snap',    'snap':  'Snap',     'Snap':  'Snap',
    'Slap Shot':  'Slap',    'slap':  'Slap',     'Slap':  'Slap',
    'Backhand':   'Backhand','backhand':'Backhand',
    'Tip-In':     'Tip-In',  'tip-in': 'Tip-In',  'Deflected': 'Tip-In', 'deflected': 'Tip-In',
    'Wrap-around':'Wrap',    'wrap-around':'Wrap',
    'Poke': 'Other', 'poke': 'Other', 'Batted': 'Other', 'bat': 'Other',
    'Penalty Shot': 'Other', 'Between Legs': 'Other', 'between-legs': 'Other',
    'Cradle': 'Other', 'cradle': 'Other',
}
# Normalize game-situation strength labels
STRENGTH_MAP = {
    '5x5': 'ES', '5v5': 'ES',
    '5x4': 'PP', '5v4': 'PP',
    '4x5': 'SH', '4v5': 'SH',
    '4x4': 'OT', '3x3': 'OT', '3v3': 'OT',
    '5x3': 'PP', '3x5': 'SH',
    '4x3': 'PP', '3x4': 'SH',
}

df = df_raw.copy()
df['shot_type_clean'] = df['shot_type'].map(SHOT_TYPE_MAP).fillna('Other')
df['situation']       = df['strength'].map(STRENGTH_MAP).fillna('Other')

# Merge xg_xgb from predictions for 2022-24 (fills dataset xg nulls)
if df_pred is not None:
    # Build a merge key that uniquely identifies rows across both frames
    merge_cols = ['season_label', 'team', 'shooter', 'x', 'y', 'goal', 'period']
    pred_xg    = df_pred[merge_cols + ['xg_xgb']].drop_duplicates(subset=merge_cols)
    df = df.merge(pred_xg, on=merge_cols, how='left')
else:
    df['xg_xgb'] = np.nan

# Unified xG: prefer dataset xg; fill 2022-24 nulls with model xg_xgb
df['xg_plot'] = df['xg'].where(df['xg'].notna(), df['xg_xgb'])

# ── Normalize to offensive zone ───────────────────────────────────────────────
# Shots with x < 0 were taken toward the x=-89 net; flip to x>0 perspective
df_off = df.copy()
mask = df_off['x'] < 0
df_off.loc[mask, 'x'] = -df_off.loc[mask, 'x']
df_off.loc[mask, 'y'] = -df_off.loc[mask, 'y']
# Keep offensive zone shots only (beyond blue line)
df_off = df_off[(df_off['x'] >= 25) & (df_off['x'] <= 100)].copy()

print(f'Offensive-zone shots: {len(df_off):,}')
print(f'Seasons: {sorted(df_off["season_label"].unique())}')
print(f'xG available for {df_off["xg_plot"].notna().sum():,} / {len(df_off):,} shots')

In [ ]:
# ── Rink drawing helpers ──────────────────────────────────────────────────────

def _circle_shape(cx, cy, r, color='red', width=1.5, fill='rgba(0,0,0,0)', layer='above'):
    return dict(type='circle', x0=cx-r, x1=cx+r, y0=cy-r, y1=cy+r,
                line=dict(color=color, width=width), fillcolor=fill, layer=layer)

def get_rink_shapes():
    """Plotly shape list for the offensive zone (x 25-100, y ±42.5)."""
    s = []

    # Ice surface background
    s.append(dict(type='rect', x0=25, x1=100, y0=-42.5, y1=42.5,
                  line=dict(color='#aac8e8', width=2),
                  fillcolor='#ddeeff', layer='below'))

    # Blue line
    s.append(dict(type='line', x0=25, x1=25, y0=-42.5, y1=42.5,
                  line=dict(color='#1565C0', width=5), layer='above'))

    # Goal line (red)
    s.append(dict(type='line', x0=89, x1=89, y0=-10.5, y1=10.5,
                  line=dict(color='#c0392b', width=2), layer='above'))

    # Net (rectangle behind goal line)
    s.append(dict(type='rect', x0=89, x1=92, y0=-3, y1=3,
                  line=dict(color='#c0392b', width=1.5),
                  fillcolor='rgba(192,57,43,0.25)', layer='above'))

    # Crease (semicircle facing toward center ice, radius=6)
    s.append(dict(type='path',
                  path='M 89,-6 A 6,6 0 0,0 89,6 Z',
                  line=dict(color='#c0392b', width=1.5),
                  fillcolor='rgba(100,149,237,0.35)', layer='above'))

    # Offensive zone face-off circles at (69, ±22), r=15
    for yc in [22, -22]:
        s.append(_circle_shape(69, yc, 15, color='#c0392b', width=1.5))
        s.append(_circle_shape(69, yc, 0.7, color='#c0392b', fill='#c0392b'))  # dot

    # Referee crease at blue line (optional cosmetic detail)
    # Hashmarks on face-off circles (small lines at 2,3,7,8 o'clock positions)

    return s


def rink_layout_2d(title='', height=540):
    """Standard 2D rink layout dict."""
    return dict(
        title=dict(text=title, font=dict(size=16)),
        shapes=get_rink_shapes(),
        xaxis=dict(range=[23, 103], title='x (ft)', showgrid=False, zeroline=False),
        yaxis=dict(range=[-44, 44], title='y (ft)', showgrid=False, zeroline=False,
                   scaleanchor='x', scaleratio=1),
        height=height,
        margin=dict(l=40, r=40, t=50, b=40),
        plot_bgcolor='rgba(0,0,0,0)',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
    )

print('Rink helpers ready.')

## 1 · Interactive 2D Shot Map

Use the controls to filter and recolor the shot chart.

| Control | Options |
|---|---|
| **Season** | Individual seasons or All |
| **Team** | Any NHL team (shooting team) |
| **Situation** | ES / PP / SH / All |
| **Color by** | xG value · Shot type · Goal/No-goal |
| **View** | Scatter · Density (hex) · Goal-rate heatmap |

In [ ]:
# ── 2D Interactive Shot Map ───────────────────────────────────────────────────

SHOT_COLORS = {
    'Wrist': '#1f77b4', 'Snap': '#ff7f0e', 'Slap': '#2ca02c',
    'Backhand': '#d62728', 'Tip-In': '#9467bd', 'Wrap': '#8c564b', 'Other': '#7f7f7f',
}

seasons_opts   = ['All'] + sorted(df_off['season_label'].unique())
teams_opts     = ['All'] + sorted(df_off['team'].dropna().unique())
sit_opts       = ['All', 'ES', 'PP', 'SH']
color_opts     = ['xG Value', 'Shot Type', 'Outcome (Goal/No-Goal)']
view_opts      = ['Scatter', 'Density Heatmap', 'Goal-Rate Heatmap']

w_season = widgets.Dropdown(options=seasons_opts, value='All',    description='Season:')
w_team   = widgets.Dropdown(options=teams_opts,   value='All',    description='Team:')
w_sit    = widgets.Dropdown(options=sit_opts,      value='All',    description='Situation:')
w_color  = widgets.Dropdown(options=color_opts,    value='xG Value', description='Color by:')
w_view   = widgets.ToggleButtons(options=view_opts, value='Scatter',
                                  style={'button_width': '160px'})
w_out    = widgets.Output()


def build_2d_map(_=None):
    with w_out:
        w_out.clear_output(wait=True)
        df = df_off.copy()

        # Apply filters
        if w_season.value != 'All':
            df = df[df['season_label'] == w_season.value]
        if w_team.value != 'All':
            df = df[df['team'] == w_team.value]
        if w_sit.value != 'All':
            df = df[df['situation'] == w_sit.value]

        title = (f"NHL Shots | Season: {w_season.value} | "
                 f"Team: {w_team.value} | Sit: {w_sit.value} | "
                 f"n={len(df):,}")

        fig = go.Figure()
        fig.update_layout(**rink_layout_2d(title=title))

        view = w_view.value

        # ── Scatter ──
        if view == 'Scatter':
            sample = df.sample(min(15_000, len(df)), random_state=42) if len(df) > 15_000 else df
            color_by = w_color.value

            if color_by == 'xG Value':
                sub = sample.dropna(subset=['xg_plot'])
                fig.add_trace(go.Scatter(
                    x=sub['x'], y=sub['y'], mode='markers',
                    marker=dict(
                        size=5, opacity=0.6,
                        color=sub['xg_plot'], colorscale='YlOrRd', showscale=True,
                        colorbar=dict(title='xG', thickness=14, len=0.6),
                        cmin=0, cmax=0.5,
                    ),
                    text=[f"Shooter: {r.shooter}<br>Team: {r.team}<br>"
                          f"Shot: {r.shot_type_clean}<br>xG: {r.xg_plot:.3f}<br>"
                          f"Goal: {'Yes' if r.goal else 'No'}"
                          for r in sub.itertuples()],
                    hoverinfo='text', name='Shots'
                ))
                # Overlay goals
                goals = sub[sub['goal'] == 1]
                fig.add_trace(go.Scatter(
                    x=goals['x'], y=goals['y'], mode='markers',
                    marker=dict(size=9, symbol='star', color='gold',
                                line=dict(color='black', width=0.5)),
                    name='Goal', hoverinfo='skip'
                ))

            elif color_by == 'Shot Type':
                for stype, color in SHOT_COLORS.items():
                    sub = sample[sample['shot_type_clean'] == stype]
                    if len(sub) == 0:
                        continue
                    fig.add_trace(go.Scatter(
                        x=sub['x'], y=sub['y'], mode='markers',
                        marker=dict(size=5, opacity=0.55, color=color),
                        name=stype
                    ))

            else:  # Outcome
                for label, val, color, sym, size in [
                    ('No Goal', 0, 'steelblue', 'circle', 4),
                    ('Goal',    1, 'crimson',   'star',   9),
                ]:
                    sub = sample[sample['goal'] == val]
                    fig.add_trace(go.Scatter(
                        x=sub['x'], y=sub['y'], mode='markers',
                        marker=dict(size=size, opacity=0.5 if val==0 else 0.8,
                                    color=color, symbol=sym,
                                    line=dict(color='black', width=0.3) if val==1 else {}),
                        name=label
                    ))

        # ── Density Heatmap ──
        elif view == 'Density Heatmap':
            fig.add_trace(go.Histogram2dContour(
                x=df['x'], y=df['y'],
                colorscale='Hot', reversescale=True,
                ncontours=18,
                contours=dict(coloring='heatmap'),
                colorbar=dict(title='Shot<br>Density', thickness=14, len=0.6),
                name='Density', showscale=True,
                xbins=dict(start=25, end=100, size=2),
                ybins=dict(start=-42.5, end=42.5, size=2),
                opacity=0.75
            ))

        # ── Goal-Rate Heatmap ──
        else:
            xb = np.linspace(25, 100, 35)
            yb = np.linspace(-42.5, 42.5, 30)
            shots_h, _, _ = np.histogram2d(df['x'], df['y'], bins=[xb, yb])
            goals_h, _, _ = np.histogram2d(
                df.loc[df['goal']==1, 'x'], df.loc[df['goal']==1, 'y'], bins=[xb, yb]
            )
            rate = np.where(shots_h >= 10, goals_h / shots_h * 100, np.nan)
            rate_smooth = gaussian_filter(np.nan_to_num(rate, nan=0), sigma=1.5)
            rate_smooth[shots_h < 10] = np.nan

            fig.add_trace(go.Heatmap(
                x=(xb[:-1] + xb[1:]) / 2,
                y=(yb[:-1] + yb[1:]) / 2,
                z=rate_smooth.T,
                colorscale='YlOrRd',
                colorbar=dict(title='Goal %', thickness=14, len=0.6),
                opacity=0.8, name='Goal Rate',
                zmin=0, zmax=25,
                hovertemplate='x=%{x:.0f} ft<br>y=%{y:.0f} ft<br>Goal Rate=%{z:.1f}%<extra></extra>'
            ))

        fig.show()


for w in [w_season, w_team, w_sit, w_color, w_view]:
    w.observe(build_2d_map, names='value')

display(widgets.VBox([
    widgets.HBox([w_season, w_team, w_sit]),
    widgets.HBox([w_color]),
    w_view,
    w_out,
]))
build_2d_map()

## 2 · 3D xG Danger Surface

The expected-goals probability is interpolated onto a regular grid and rendered as a 3D terrain.
The net area becomes a "volcano" of danger — drag to rotate, scroll to zoom.

In [ ]:
# ── 3D xG Danger Surface ─────────────────────────────────────────────────────

df_xg = df_off.dropna(subset=['xg_plot']).copy()

# Bin mean xG onto a grid
x_bins = np.linspace(25, 100, 60)
y_bins = np.linspace(-42.5, 42.5, 50)

xg_grid, _, _, _ = binned_statistic_2d(
    df_xg['x'], df_xg['y'], df_xg['xg_plot'],
    statistic='mean', bins=[x_bins, y_bins]
)
count_grid, _, _, _ = binned_statistic_2d(
    df_xg['x'], df_xg['y'], df_xg['xg_plot'],
    statistic='count', bins=[x_bins, y_bins]
)

# Suppress low-sample bins, smooth
xg_grid[count_grid < 20] = np.nan
xg_smooth = gaussian_filter(np.nan_to_num(xg_grid, nan=0), sigma=2.0)
xg_smooth[count_grid < 20] = np.nan

xc = (x_bins[:-1] + x_bins[1:]) / 2
yc = (y_bins[:-1] + y_bins[1:]) / 2

fig3d_surf = go.Figure()

# Main xG surface
fig3d_surf.add_trace(go.Surface(
    x=xc, y=yc, z=(xg_smooth * 100).T,   # z in xG %
    colorscale='YlOrRd',
    colorbar=dict(title='xG %', thickness=16, len=0.65),
    opacity=0.88,
    hovertemplate='x=%{x:.0f} ft<br>y=%{y:.0f} ft<br>xG=%{z:.1f}%<extra></extra>',
    name='xG Surface',
))

# Flat ice-level reference plane at z=0
xx, yy = np.meshgrid(np.linspace(25, 100, 20), np.linspace(-42.5, 42.5, 20))
fig3d_surf.add_trace(go.Surface(
    x=xx.T, y=yy.T, z=np.zeros_like(xx.T),
    colorscale=[[0, '#b0d4f0'], [1, '#b0d4f0']],
    showscale=False, opacity=0.25, name='Ice Surface',
    hoverinfo='skip'
))

# Net marker at (89, 0)
net_z = float(np.nanmax(xg_smooth) * 100) + 2
fig3d_surf.add_trace(go.Scatter3d(
    x=[89], y=[0], z=[net_z],
    mode='markers+text',
    marker=dict(size=10, color='black', symbol='diamond'),
    text=['Net'], textposition='top center',
    showlegend=False
))

# Blue line bar at x=25
fig3d_surf.add_trace(go.Scatter3d(
    x=[25, 25], y=[-42.5, 42.5], z=[0, 0],
    mode='lines', line=dict(color='blue', width=5),
    name='Blue Line', showlegend=False
))

fig3d_surf.update_layout(
    title=dict(text='3D xG Danger Surface — Offensive Zone', font=dict(size=17)),
    scene=dict(
        xaxis=dict(title='x (ft)', range=[25, 100]),
        yaxis=dict(title='y (ft)', range=[-42.5, 42.5]),
        zaxis=dict(title='xG %', range=[0, None]),
        camera=dict(
            eye=dict(x=-1.8, y=0.0, z=1.2),
            up=dict(x=0, y=0, z=1)
        ),
        aspectratio=dict(x=2, y=2, z=0.7),
    ),
    height=620,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig3d_surf.show()

## 3 · 3D Shot Scatter (z = xG)

Each shot is a point in 3D space: x/y = ice coordinates, z = xG value assigned by the model.
Goals (red stars) float high; routine perimeter shots cluster near the ice surface.
Drag to rotate — you can clearly see the slot danger zone rising above the rest.

In [ ]:
# ── 3D Shot Scatter ────────────────────────────────────────────────────────────

rng = np.random.default_rng(42)

# Sample for performance — keep all goals, sample non-goals
df_goals   = df_xg[df_xg['goal'] == 1]
df_ngoals  = df_xg[df_xg['goal'] == 0]

n_goal     = min(len(df_goals), 5_000)
n_nongoal  = min(len(df_ngoals), 12_000)

g_idx  = rng.choice(len(df_goals),  n_goal,   replace=False)
ng_idx = rng.choice(len(df_ngoals), n_nongoal, replace=False)

gs  = df_goals.iloc[g_idx]
ngs = df_ngoals.iloc[ng_idx]

fig3d_sc = go.Figure()

# Non-goal shots (blue, small, semi-transparent)
fig3d_sc.add_trace(go.Scatter3d(
    x=ngs['x'], y=ngs['y'], z=ngs['xg_plot'],
    mode='markers',
    marker=dict(size=2, color='steelblue', opacity=0.25,
                colorscale=None),
    name='No Goal',
    text=[f"{r.shooter}<br>{r.shot_type_clean}<br>xG={r.xg_plot:.3f}"
          for r in ngs.itertuples()],
    hoverinfo='text'
))

# Goals (red stars, larger)
fig3d_sc.add_trace(go.Scatter3d(
    x=gs['x'], y=gs['y'], z=gs['xg_plot'],
    mode='markers',
    marker=dict(size=5, color='crimson', opacity=0.85,
                symbol='diamond', line=dict(color='white', width=0.5)),
    name='Goal',
    text=[f"{r.shooter}<br>{r.shot_type_clean}<br>xG={r.xg_plot:.3f}"
          for r in gs.itertuples()],
    hoverinfo='text'
))

# Flat ice surface
fig3d_sc.add_trace(go.Surface(
    x=xx.T, y=yy.T, z=np.zeros_like(xx.T),
    colorscale=[[0, '#b0d4f0'], [1, '#b0d4f0']],
    showscale=False, opacity=0.20, name='Ice',
    hoverinfo='skip'
))

# Blue line
fig3d_sc.add_trace(go.Scatter3d(
    x=[25, 25], y=[-42.5, 42.5], z=[0, 0],
    mode='lines', line=dict(color='blue', width=4),
    showlegend=False
))

fig3d_sc.update_layout(
    title=dict(text='3D Shot Scatter — z = xG Value', font=dict(size=17)),
    scene=dict(
        xaxis=dict(title='x (ft)', range=[25, 100]),
        yaxis=dict(title='y (ft)', range=[-42.5, 42.5]),
        zaxis=dict(title='xG', range=[0, 0.7]),
        camera=dict(
            eye=dict(x=-1.8, y=0.1, z=1.0),
            up=dict(x=0, y=0, z=1)
        ),
        aspectratio=dict(x=2, y=2, z=0.85),
    ),
    legend=dict(x=0.75, y=0.95),
    height=640,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig3d_sc.show()

## Tips

- **Rotate 3D plots** — click and drag
- **Zoom** — scroll wheel or pinch
- **Hover** — see shooter name, shot type, xG on all scatter traces
- **Toggle traces** — click legend items to show/hide
- **Export** — camera icon (top-right of each plot) → Download PNG

### Interesting filters to try
| Filter | What to look for |
|---|---|
| Situation = PP | Wider spread, more slot shots |
| Situation = SH | Heavy concentration of perimeter shots |
| Team = EDM | Connor McDavid shot distribution |
| View = Goal-Rate Heatmap | The slot danger zone becomes very clear |